In [2]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTENC
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline
from sklearn.metrics import recall_score
from sklearn.metrics import make_scorer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin


In [3]:
# Importing and dropping unwanted columns
df = pd.read_csv('/Users/Tudor.Orosz/Documents/School/Thesis/Extract Data Athena/Dataframe marketing and platform.csv')

df = df.drop(["id", "accountid", "account_sales_unit__c", "sales_unit_group__c", "name", "createddate"], axis = 1)

df

,gtm_segment__c,gtm_industry__c,region__c,country_grouping__c,opportunityid,M_WBN_Attended,M_WBN_Attended On-Demand,M_Other_Filled Out Form,M_TS_Attended,M_TS_Visited Booth,...,P_VisitCloudProvidertAlerts,P_WindowsServiceDownloaded,P_WmBuzzPostMention,P_WmDeployConsistencyError,P_WmDeployFailed,P_WmDeployStarted,P_WmDeploySucceeded,P_WmIntroTutorialBroken,P_WmIntroTutorialStep,P_WmProjectOpened
0,MXD,Business Services,EMEA - REV,Benelux,0,0,0,0,0,0,...,0,0,0,8,1,22,12,0,50,12
1,MXD,Government,EMEA - PUBLIC,UKI,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,16
2,MXD,Software & IT,US - Diversified,US,0,0,0,0,0,0,...,0,0,0,0,0,5,5,0,44,19
3,MXD,Software & IT,APAC - SEA,SEA,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,6
4,MXD,Software & IT,EMEA - REV,Benelux,0,0,0,0,0,0,...,0,0,0,0,0,2,1,0,5,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52150,CH,Clearing House Review,US - Other,US,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52151,MXD,Software & IT,APAC - G. China,G. China,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,31,5
52152,MXD,Software & IT,US - Diversified,US,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52153,MXD,Business Services,EMEA - REV,Middle East,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# One hot encoding the categorical cols

categorical_cols = ['gtm_segment__c', 'gtm_industry__c', 'region__c', 'country_grouping__c']
df_encoded = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols, prefix_sep=' ', dummy_na=False)
df_encoded = df_encoded.astype(int)
# Display the resulting dataframe
df = df_encoded
df_encoded

,opportunityid,M_WBN_Attended,M_WBN_Attended On-Demand,M_Other_Filled Out Form,M_TS_Attended,M_TS_Visited Booth,M_LE_Attended,M_WBN_Attended On-demand,M_VLE_Attended,M_TS_Visited booth,...,country_grouping__c Japan,country_grouping__c Middle East,country_grouping__c Nordics,country_grouping__c RoE,country_grouping__c SEA,country_grouping__c South Africa,country_grouping__c South Korea,country_grouping__c Spain,country_grouping__c UKI,country_grouping__c US
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52150,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
52151,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
52152,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
52153,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [5]:
# num_cols = df.shape[1]
# print(num_cols)

In [6]:
X = df.drop("opportunityid", axis = 1)
y = df.loc[:, ["opportunityid"]]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=548)

X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=548)

In [8]:
# Define the columns to be one-hot encoded
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Specify numerical variables
X_numb = X_train.select_dtypes(include=['int64', 'float64'])

# Define PCA indices
indices_pca = [i for i, col in enumerate(X.columns) if col in X_numb.columns]

In [9]:
# Define the pipeline
numeric_transformer = Pipeline(steps=[
    #('imputer', KNNImputer()), # for missing values
    ('scaler', StandardScaler()), # standardizing
    #('scaler', MinMaxScaler()), # normalizing
    #('to_df', FunctionTransformer(lambda x: pd.DataFrame(x, columns=X_train2.select_dtypes(include=['int64', 'float64']).columns)))
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, X_train2.select_dtypes(include=['int64', 'float64']).columns.tolist()),
        # ('cat', categorical_transformer, categorical_cols),
        # ("pca", PCA(random_state=548, n_components=25), indices_pca),
        # ('smotenc', SmoteNCWrapper(categorical_features=[1, 2], random_state=548)),
    ],
    remainder='passthrough'
    )

# Add the upsampler and the pipeline to a machine learning model
model = Pipeline(steps=[
    # ('over', SMOTE(random_state=548)),
    # ('smotenc', SmoteNCWrapper(categorical_features=[1, 2], random_state=548)),
    ('preprocessor', preprocessor),
    ('regressor', XGBClassifier(random_state=548))
])

parameters = {  'regressor__n_estimators': [1000],
    'regressor__max_depth': [5],
    'regressor__learning_rate': [0.01],
    # 'regressor__num_leaves': [31],
    # 'regressor__min_child_samples': [20],
    'regressor__reg_alpha': [0.1], 
    'regressor__reg_lambda': [0.1]
}

In [10]:
# Set up the grid search object
grid_search = GridSearchCV(model, parameters, cv=10, scoring=make_scorer(recall_score))

In [11]:
categorical_cols = X_train2.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

[]


In [12]:
# Fit the grid search object to the training data
grid_search.fit(X_train2, y_train2.values.ravel())

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['M_WBN_Attended',
                                                                          'M_WBN_Attended '
                                                                          'On-Demand',
                                                                          'M_Other_Filled '
                                                                          'Out '
                                                                          'Form',
                                                                          'M_TS_Attended',
                                                                          'M_TS_Visited '
                                                                          'Booth',
                                                                          'M_LE_Attended',
                                                                          'M_WBN_Attended '
                                                                          'On-demand',
                                                                          'M_VLE_Attended',
                                                                          'M_TS_V...
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      n_estimators=100,
                                                      n_jobs=None,
                                                      num_parallel_tree=None,
                                                      predictor=None,
                                                      random_state=548, ...))]),
             param_grid={'regressor__learning_rate': [0.01],
                         'regressor__max_depth': [5],
                         'regressor__n_estimators': [1000],
                         'regressor__reg_alpha': [0.1],
                         'regressor__reg_lambda': [0.1]},
             scoring=make_scorer(recall_score))

In [13]:
# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(best_params)

{'regressor__learning_rate': 0.01, 'regressor__max_depth': 5, 'regressor__n_estimators': 1000, 'regressor__reg_alpha': 0.1, 'regressor__reg_lambda': 0.1}


In [14]:
# Feature importance
hd = list(X_train2.columns)

# Calculate the importances
feature_importances = grid_search.best_estimator_.named_steps["regressor"].feature_importances_
feature_scores = list(zip(hd, feature_importances))
df_feature_importances = pd.DataFrame(feature_scores, columns=['Feature', 'Importance Score (%)'])

# Should add up o 100%
total_score = df_feature_importances['Importance Score (%)'].sum()
print(total_score)

1.0


In [15]:
# Import and view dataframe
# df_feature_importances.to_csv('feature_importances.csv', index=False)
df_feature_importances
# --------------------------------end feature importance------------------------------

,Feature,Importance Score (%)
0,M_WBN_Attended,0.013391
1,M_WBN_Attended On-Demand,0.003774
2,M_Other_Filled Out Form,0.000000
3,M_TS_Attended,0.005105
4,M_TS_Visited Booth,0.000000
...,...,...
379,country_grouping__c South Africa,0.000000
380,country_grouping__c South Korea,0.000000
381,country_grouping__c Spain,0.000000
382,country_grouping__c UKI,0.002863


In [16]:
y_pred = grid_search.predict(X_val)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [26]:
y_proba = grid_search.predict_proba(X_val)
y_pred_new = (y_proba[:, 1] > 0.2).astype(int)

In [27]:
# Compute the confusion matrix
cm = confusion_matrix(y_val, y_pred_new)

# Print the confusion matrix with labels
print("Confusion matrix:")
print("           Predicted")
print("           0   1")
print("Actual 0  ", cm[0][0], " ", cm[0][1])
print("       1  ", cm[1][0], " ", cm[1][1])

Confusion matrix:
           Predicted
           0   1
Actual 0   9560   591
       1   379   423


In [28]:
# Calculate the confusion matrix
cm = confusion_matrix(y_val, y_pred_new)

# Extract the true negatives, false positives, false negatives, and true positives
tn, fp, fn, tp = cm.ravel()

# Calculate accuracy, specificity, and sensitivity
accuracy = (tp + tn) / (tp + tn + fp + fn)
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

# Print the results
print(f"Accuracy: {accuracy:.2f}")
print(f"Specificity: {specificity:.2f}")
print(f"Sensitivity: {sensitivity:.2f}")

Accuracy: 0.91
Specificity: 0.94
Sensitivity: 0.53


In [1]:
from sklearn.metrics import f1_score, roc_auc_score

# Calculate the F1 score
f1 = f1_score(y_val, y_pred_new)
print(f"F1 score: {f1:.2f}")

# Calculate the AUC
auc = roc_auc_score(y_val, y_proba[:, 1])
print(f"AUC: {auc:.2f}")

NameError: name 'y_val' is not defined

In [20]:
df_feature_importances

,Feature,Importance Score (%)
0,M_WBN_Attended,0.013391
1,M_WBN_Attended On-Demand,0.003774
2,M_Other_Filled Out Form,0.000000
3,M_TS_Attended,0.005105
4,M_TS_Visited Booth,0.000000
...,...,...
379,country_grouping__c South Africa,0.000000
380,country_grouping__c South Korea,0.000000
381,country_grouping__c Spain,0.000000
382,country_grouping__c UKI,0.002863


In [21]:
df_filtered = df_feature_importances[df_feature_importances['Importance Score (%)'] != 0]

In [22]:
df_filtered

,Feature,Importance Score (%)
0,M_WBN_Attended,0.013391
1,M_WBN_Attended On-Demand,0.003774
3,M_TS_Attended,0.005105
5,M_LE_Attended,0.010816
6,M_WBN_Attended On-demand,0.002286
...,...,...
370,country_grouping__c France,0.004768
375,country_grouping__c Middle East,0.005372
376,country_grouping__c Nordics,0.005342
382,country_grouping__c UKI,0.002863
